# 🚀 The Honest Ring Benchmark: AdamV vs AdamW (Tartarus Arena)
Welcome to the official Kaggle benchmark for **AdamV** (Adam Vedic Optimizer).
This is the **Tartarus Arena**, an extreme stress test featuring a 100-layer MLP without residual connections.

### 1. JIT CUDA Compilation (The Fused Kernel)
Instead of running on pure Python and suffering from CPU-GPU sync overhead, we use PyTorch's `load_inline` to compile our Bakhshali Quartic Brake kernel directly on Kaggle's Nvidia GPUs on-the-fly!

In [ ]:
import torch
import torch.utils.cpp_extension
import time

print("🔥 Compilando o Fused CUDA Kernel do AdamV (JIT)... Isso levará cerca de 40 segundos na primeira vez.")
start_time = time.time()

cuda_source = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <cmath>
#include <ATen/cuda/CUDAContext.h>

#ifndef M_PI
#define M_PI 3.14159265358979323846
#endif

const int BLOCK_SIZE = 256;

template <typename scalar_t>
__global__ void adamv_prepare_kernel(
    float* __restrict__ exp_avg,
    float* __restrict__ exp_avg_sq,
    float* __restrict__ direcao_buffer,
    const scalar_t* __restrict__ grad,
    float beta1, float beta2, float bias_correction2, float eps, int step, int numel) {
    
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < numel) {
        float g = static_cast<float>(grad[idx]);
        float m = exp_avg[idx];
        float v = exp_avg_sq[idx];

        v = beta2 * v + (1.0f - beta2) * g * g;
        exp_avg_sq[idx] = v;

        float v_hat = v / bias_correction2;
        float sqrt_v_hat = sqrt(v_hat);

        // BRCM: Excess Shock Isolator (delta)
        // Tighter tuning: 1.0x margin to prevent excess momentum in sharp minima
        float delta = fmaxf(0.0f, std::abs(g) - 1.5f * sqrt_v_hat);
        float denom_brcm = sqrt_v_hat + delta + eps;
        float bakh_residual = (delta * delta) / (2.0f * denom_brcm);
        
        // Full curvature shift to avoid carrying too much momentum
        float curvature_shift = bakh_residual / (sqrt_v_hat + eps);
        float beta1_dynamic = beta1 * std::exp(-0.5f * curvature_shift);
        
        m = beta1_dynamic * m + (1.0f - beta1_dynamic) * g;
        exp_avg[idx] = m;
        
        float bias_correction1_dynamic = 1.0f - std::pow(beta1_dynamic, static_cast<float>(step));

        float m_hat = m / bias_correction1_dynamic;
        
        direcao_buffer[idx] = m_hat / (sqrt_v_hat + eps);
    }
}

template <typename scalar_t>
__global__ void adamv_update_kernel(
    scalar_t* __restrict__ params,
    const scalar_t* __restrict__ grad,
    float* __restrict__ exp_avg,
    float* __restrict__ exp_avg_sq,
    const float* __restrict__ direcao_buffer,
    const float* __restrict__ norm_tensor_ptr,
    float progresso, float cooling_factor, float bakh_thresh_eff, float bias_correction2, float eps, 
    float wd_factor, float lr_max, float weight_decay, int numel, int D, int step,
    bool enable_cooling, bool enable_brake,
    bool omni_triggered, uint32_t punning_mask) {
    
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < numel) {
        scalar_t p = params[idx];
        
        float g = static_cast<float>(grad[idx]);
        float dir = direcao_buffer[idx];
        float v = exp_avg_sq[idx];
        
        float lr_efetivo = lr_max;
        if (enable_cooling) {
            float norm_dir = (*norm_tensor_ptr) / sqrt(static_cast<float>(D));
            float envelope = (1.0f + progresso) / (progresso + norm_dir + eps);
            lr_efetivo = lr_max * min(envelope * cooling_factor, 1.5f);
        }
        
        float a = lr_efetivo * dir;
        float v_hat = v / bias_correction2;
        float sqrt_v = sqrt(v_hat);
        
        float step_size = a;
        if (enable_brake) {
            bool explosao_mask = std::abs(g) > (bakh_thresh_eff * sqrt_v);
            if (explosao_mask) {
                // Dimensional Mismatch Fix: parameter absolute magnitude
                // Tightened Tuning: Stronger brake penalty for late-stage stability
                float denom = std::abs(static_cast<float>(p)) + (std::abs(a) * 2.0f) + eps;
                float correction = (a * a) / (2.0f * denom);
                float bakhshali_brake = a - copysignf(1.0f, a) * correction;
                step_size = bakhshali_brake;
            }
        }
        
        // Orthogonal Truncated Levy Flight Injection
        uint32_t seed = (static_cast<uint32_t>(step) * 31337) ^ static_cast<uint32_t>(idx);
        seed = (seed * 1664525) + 1013904223;
        float u = (static_cast<float>(seed) / 4294967296.0f) * 0.999f + 0.0005f;
        float cauchy = tanf(M_PI * (u - 0.5f));
        cauchy = fmaxf(-500.0f, fminf(500.0f, cauchy));
        float levy_scale = sqrt_v / (sqrt_v + 0.1f);
        float levy_mult = 0.01f * fmaxf(0.0f, 1.0f - progresso);
        float levy_jump = lr_efetivo * levy_mult * cauchy * levy_scale;
        
        step_size -= levy_jump;
        
        if (weight_decay != 0.0f) {
            p = static_cast<scalar_t>(static_cast<float>(p) * (1.0f - lr_max * weight_decay * wd_factor));
        }
        
        params[idx] = static_cast<scalar_t>(static_cast<float>(p) - step_size);
        
        if (omni_triggered) {
            if (sizeof(scalar_t) == 4) {
                float p_new = static_cast<float>(params[idx]);
                uint32_t p_int = __float_as_uint(p_new);
                
                uint32_t sign = p_int & 0x80000000;
                uint32_t exp  = p_int & 0x7F800000;
                uint32_t mant = p_int & 0x007FFFFF;
                
                uint32_t mant_mod = (((mant + 1) * 31337) & 0x007FFFFF) & punning_mask;
                
                p_new = __uint_as_float(sign | exp | mant_mod);
                params[idx] = static_cast<scalar_t>(p_new);
            }
            exp_avg[idx] = 0.0f;
            exp_avg_sq[idx] *= 0.1f;
        }
    }
}

#define CHECK_CUDA(x) TORCH_CHECK(x.device().is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

void adamv_step_cuda(
    at::Tensor p,
    at::Tensor grad,
    at::Tensor exp_avg,
    at::Tensor exp_avg_sq,
    at::Tensor direcao,
    float lr,
    float beta1,
    float beta2,
    float eps,
    float weight_decay,
    float progresso,
    float bakh_thresh_eff,
    int step,
    int D,
    bool enable_cooling,
    bool enable_brake,
    bool omni_triggered,
    int64_t punning_mask) 
{
    CHECK_INPUT(p);
    CHECK_INPUT(grad);
    CHECK_INPUT(exp_avg);
    CHECK_INPUT(exp_avg_sq);
    CHECK_INPUT(direcao);

    int numel = p.numel();
    int blocks = (numel + BLOCK_SIZE - 1) / BLOCK_SIZE;

    float bias_correction2 = 1.0f - std::pow(static_cast<float>(beta2), static_cast<float>(step));

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(p.scalar_type(), "adamv_prepare", [&] {
        adamv_prepare_kernel<scalar_t><<<blocks, BLOCK_SIZE, 0, stream>>>(
            exp_avg.data_ptr<float>(),
            exp_avg_sq.data_ptr<float>(),
            direcao.data_ptr<float>(),
            grad.data_ptr<scalar_t>(),
            beta1, beta2, bias_correction2, eps, step, numel
        );
    });

    // Compute norm asynchronously on GPU
    at::Tensor norm_tensor = at::linalg_norm(direcao);
    
    float cooling_factor;
    if (progresso < 0.1f) {
        cooling_factor = 0.01f + (progresso / 0.1f) * 0.99f;
    } else {
        float cos_progresso = (progresso - 0.1f) / 0.9f;
        cooling_factor = 1.0f;
    }
    float wd_factor = 0.5f * (1.0f + std::cos(M_PI * progresso));

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(p.scalar_type(), "adamv_update", [&] {
        adamv_update_kernel<scalar_t><<<blocks, BLOCK_SIZE, 0, stream>>>(
            p.data_ptr<scalar_t>(),
            grad.data_ptr<scalar_t>(),
            exp_avg.data_ptr<float>(),
            exp_avg_sq.data_ptr<float>(),
            direcao.data_ptr<float>(),
            norm_tensor.data_ptr<float>(),
            progresso, cooling_factor, bakh_thresh_eff, bias_correction2, eps, wd_factor, lr, weight_decay, numel, D, step,
            enable_cooling, enable_brake,
            omni_triggered, static_cast<uint32_t>(punning_mask)
        );
    });
}

"""

cpp_source = """
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>

void adamv_step_cuda(at::Tensor p, at::Tensor grad, at::Tensor exp_avg, at::Tensor exp_avg_sq, at::Tensor direcao, float lr, float beta1, float beta2, float eps, float weight_decay, float progresso, float bakh_thresh_eff, int step, int D, bool enable_cooling, bool enable_brake, bool omni_triggered, int64_t punning_mask);
"""

adamv_cuda_module = torch.utils.cpp_extension.load_inline(
    name='adamv_cuda_jit',
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=['adamv_step_cuda'],
    with_cuda=True,
    extra_cflags=['-O3'],
    extra_cuda_cflags=['-O3', '-use_fast_math']
)
print(f"✅ CUDA Kernel compilado com sucesso em {time.time()-start_time:.2f} segundos!")



### 2. The AdamV Optimizer Source Code
We inject the Python wrapper of AdamV directly into this notebook. It will automatically detect the `adamv_cuda_module` compiled above and route all GPU tensors to it.

In [ ]:
import math
from torch.optim.optimizer import Optimizer
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math
import torch
import math

import torch
import math

class AdamV(torch.optim.Optimizer):
    """
    AdamV (Adam Vedic) Optimizer - Pure Python Version.
    AdamV 3.1: Harmonic Refactor (In-Place VRAM Opt, OMNI State Fix, Modular Flags)
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, 
                 weight_decay=0.01, total_steps=10000, 
                 bakhshali_threshold=15.0, enable_omni=True,
                 lp_kappa=0.1, lp_omega=10.0, punning_mask=0xFFFFE000,
                 enable_ignition=False, enable_cooling=True, enable_brake=True):
                 
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        total_steps=total_steps, bakhshali_threshold=bakhshali_threshold,
                        enable_omni=enable_omni, lp_kappa=lp_kappa, lp_omega=lp_omega, 
                        punning_mask=punning_mask,
                        enable_ignition=enable_ignition, enable_cooling=enable_cooling, enable_brake=enable_brake)
        super(AdamV, self).__init__(params, defaults)
        
        if 'omni_loss_ema' not in self.param_groups[0]:
            self.param_groups[0]['omni_loss_ema'] = float('inf')
            self.param_groups[0]['omni_patience'] = 0.0
            self.param_groups[0]['omni_clock_reset_step'] = 0
            self.param_groups[0]['omni_global_step'] = 0

    @torch.no_grad()
    def step(self, closure=None, current_loss=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
                
        if current_loss is not None:
            loss = current_loss

        g_state = self.param_groups[0]
        g_state['omni_global_step'] += 1
        current_step = g_state['omni_global_step']
        
        omni_triggered = False
        if loss is not None and len(self.param_groups) > 0 and self.param_groups[0]['enable_omni']:
            loss_val = float(loss) if isinstance(loss, torch.Tensor) else loss
            if g_state['omni_loss_ema'] == float('inf'):
                g_state['omni_loss_ema'] = loss_val
                g_state['omni_patience'] = 0.0
            else:
                g_state['omni_loss_ema'] = 0.9 * g_state['omni_loss_ema'] + 0.1 * loss_val
                
            is_worse = loss_val > g_state['omni_loss_ema'] * 0.99
            g_state['omni_patience'] = g_state['omni_patience'] + 1.0 if is_worse else 0.0
            
            patience_limit = max(500, int(self.param_groups[0]['total_steps'] * 0.05))
            if g_state['omni_patience'] >= patience_limit:
                omni_triggered = True
                g_state['omni_patience'] = 0.0
                
        for group in self.param_groups:
            lr_max = group['lr']
            total_steps = group['total_steps']
            
            # Autonomous Ignition
            if group.get('enable_ignition', True):
                ignition = min(1.0, current_step / max(1.0, total_steps * 0.10))
                lr_max = lr_max * ignition
                
            # Diophantine QMC Jitter (Time-Domain Scrambling)
            # Adds a low-discrepancy jitter to the base learning rate
            jitter_scale = 0.15 * max(0.0, 1.0 - (current_step / max(1, total_steps)))
            lr_max = lr_max * (1.0 + jitter_scale * ((current_step * 0.6180339887) % 1.0 - 0.5))

            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']
            total_steps = group['total_steps']
            bakh_thresh = group['bakhshali_threshold']
            lp_kappa = group['lp_kappa']
            lp_omega = group['lp_omega']
            punning_mask = group['punning_mask']
            
            internal_step = current_step - g_state['omni_clock_reset_step']
            progresso = min(1.0, internal_step / max(1, total_steps))

            # Resfriamento MacroscÃ³pico (Cosine Decay)
            decay_cosseno = 0.01 + 0.99 * 0.5 * (1.0 + math.cos(math.pi * progresso))
            
            # Log-Periodic Topological Cooling (Nyquist Capped)
            onda = 0.3 * math.cos(math.pi * 4.0 * progresso)
            
            # O envelope total funde o decaimento com a onda
            lr_max = lr_max * decay_cosseno * (1.0 + onda)

            
            LP_Fator = 1.0 + lp_kappa * math.cos(lp_omega * math.log(1.0 + progresso * 10.0))
            bakh_thresh_eff = bakh_thresh * LP_Fator
            
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                
                # 100% In-Place BRCM
                grad_f = grad.float()
                
                # Update exp_avg_sq first!
                exp_avg_sq.mul_(beta2).addcmul_(grad_f, grad_f, value=1.0 - beta2)
                
                bias_correction2 = 1.0 - beta2 ** state['step']
                v_hat = exp_avg_sq / bias_correction2
                sqrt_v_hat = v_hat.sqrt()
                
                # Excess Shock Isolator (delta)
                # Tighter tuning: 1.0x margin to prevent excess momentum in sharp minima
                delta = torch.clamp(torch.abs(grad_f) - 1.5 * sqrt_v_hat, min=0.0)
                
                denom_brcm = sqrt_v_hat + delta + eps
                bakh_residual = (delta * delta) / (denom_brcm * 2.0)
                
                # Full curvature shift to avoid carrying too much momentum
                curvature_shift = bakh_residual / (sqrt_v_hat + eps)
                beta1_eff = torch.exp(-0.5 * curvature_shift) * beta1
                
                bias_correction1_dynamic = 1.0 - beta1_eff ** state['step']
                
                # Update momentum In-Place
                exp_avg.mul_(beta1_eff).add_(grad_f * (1.0 - beta1_eff))
                
                direcao = (exp_avg / bias_correction1_dynamic) / (sqrt_v_hat + eps)
                
                norm_dir_padrao = torch.linalg.norm(direcao) / math.sqrt(p.numel())
                
                if group.get('enable_cooling', False):
                    envelope = (1.0 + progresso) / (progresso + norm_dir_padrao + eps)
                    if progresso < 0.1:
                        cooling_factor = 0.01 + (progresso / 0.1) * 0.99
                    else:
                        cos_progresso = (progresso - 0.1) / 0.9
                        cooling_factor = 1.0
                    lr_efetivo = lr_max * torch.clamp(envelope * cooling_factor, max=1.5)
                else:
                    lr_efetivo = lr_max
                
                a = direcao.mul_(lr_efetivo)
                
                explosao_mask = torch.abs(grad) > (bakh_thresh_eff * sqrt_v_hat)
                
                # Dimensional Mismatch Fix: parameter absolute magnitude
                # Tightened Tuning: Stronger brake penalty for late-stage stability
                denom = p.abs() + (a.abs() * 2.0) + eps
                correction = (a * a) / (denom * 2.0)
                
                bakhshali_brake = a - (torch.sign(a) * correction)
                
                if group.get('enable_brake', True):
                    step_size = torch.where(explosao_mask, bakhshali_brake, a)
                else:
                    step_size = a
                    
                # Orthogonal Truncated Levy Flight Injection
                u = torch.rand_like(p) * 0.999 + 0.0005
                cauchy = torch.tan(math.pi * (u - 0.5))
                cauchy = torch.clamp(cauchy, -500.0, 500.0)
                levy_scale = sqrt_v_hat / (sqrt_v_hat + 0.1)
                levy_mult = 0.01 * max(0.0, 1.0 - progresso)
                levy_jump = lr_efetivo * levy_mult * cauchy * levy_scale
                step_size.sub_(levy_jump)
                
                if weight_decay != 0:
                    wd_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                    p.mul_(1.0 - lr_max * weight_decay * wd_factor)
                    
                p.sub_(step_size)
                
                if omni_triggered:
                    if p.dtype == torch.float32:
                        p_int = p.view(torch.int32)
                        sign_exp = p_int & 0xFF800000
                        mant = p_int & 0x007FFFFF
                        
                        mask_val = int(punning_mask)
                        if mask_val > 0x7FFFFFFF:
                            mask_val -= 0x100000000
                        
                        scrambled_mant = (((mant + 1) * 31337) & 0x007FFFFF) & mask_val
                        p_new = (sign_exp | scrambled_mant).view(torch.float32)
                        p.copy_(p_new)
                    
                    state['exp_avg'].zero_()
                    state['exp_avg_sq'].mul_(0.1)
                    state['step'] = 0

        if omni_triggered:
            g_state['omni_clock_reset_step'] = current_step
                
        return loss

class AdamVCpp(torch.optim.Optimizer):
    """
    AdamV (Adam Vedic) Optimizer - C++ Fused Kernel Version.
    AdamV 3.1: Harmonic Refactor
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, 
                 weight_decay=0.01, total_steps=10000, 
                 bakhshali_threshold=15.0, enable_omni=True,
                 lp_kappa=0.1, lp_omega=10.0, punning_mask=0xFFFFE000,
                 enable_ignition=False, enable_cooling=True, enable_brake=True):
                 
        self.adamv_cpp = None
            
        self.adamv_cuda = adamv_cuda
            
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
            
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        total_steps=total_steps, bakhshali_threshold=bakhshali_threshold,
                        enable_omni=enable_omni, lp_kappa=lp_kappa, lp_omega=lp_omega, 
                        punning_mask=punning_mask,
                        enable_ignition=enable_ignition, enable_cooling=enable_cooling, enable_brake=enable_brake)
        super(AdamVCpp, self).__init__(params, defaults)
        
        if 'omni_loss_ema' not in self.param_groups[0]:
            self.param_groups[0]['omni_loss_ema'] = float('inf')
            self.param_groups[0]['omni_patience'] = 0.0
            self.param_groups[0]['omni_clock_reset_step'] = 0
            self.param_groups[0]['omni_global_step'] = 0

    @torch.no_grad()
    def step(self, closure=None, current_loss=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        if current_loss is not None:
            loss = current_loss

        g_state = self.param_groups[0]
        g_state['omni_global_step'] += 1
        current_step = g_state['omni_global_step']
        
        omni_triggered = False
        if loss is not None and len(self.param_groups) > 0 and self.param_groups[0]['enable_omni']:
            loss_val = float(loss) if isinstance(loss, torch.Tensor) else loss
            if g_state['omni_loss_ema'] == float('inf'):
                g_state['omni_loss_ema'] = loss_val
                g_state['omni_patience'] = 0.0
            else:
                g_state['omni_loss_ema'] = 0.9 * g_state['omni_loss_ema'] + 0.1 * loss_val
                
            is_worse = loss_val > g_state['omni_loss_ema'] * 0.99
            g_state['omni_patience'] = g_state['omni_patience'] + 1.0 if is_worse else 0.0
            
            patience_limit = max(500, int(self.param_groups[0]['total_steps'] * 0.05))
            if g_state['omni_patience'] >= patience_limit:
                omni_triggered = True
                g_state['omni_patience'] = 0.0
                
        for group in self.param_groups:
            lr_max = group['lr']
            total_steps = group['total_steps']
            
            if group.get('enable_ignition', True):
                ignition = min(1.0, current_step / max(1.0, total_steps * 0.10))
                lr_max = lr_max * ignition
                
            # Diophantine QMC Jitter (Time-Domain Scrambling)
            jitter_scale = 0.15 * max(0.0, 1.0 - (current_step / max(1, total_steps)))
            lr_max = lr_max * (1.0 + jitter_scale * ((current_step * 0.6180339887) % 1.0 - 0.5))

            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']
            total_steps = group['total_steps']
            bakh_thresh = group['bakhshali_threshold']
            lp_kappa = group['lp_kappa']
            lp_omega = group['lp_omega']
            punning_mask = group['punning_mask']
            
            internal_step = current_step - g_state['omni_clock_reset_step']
            progresso = min(1.0, internal_step / max(1, total_steps))

            # Resfriamento MacroscÃ³pico (Cosine Decay)
            decay_cosseno = 0.01 + 0.99 * 0.5 * (1.0 + math.cos(math.pi * progresso))
            
            # Log-Periodic Topological Cooling (Nyquist Capped)
            onda = 0.3 * math.cos(math.pi * 4.0 * progresso)
            
            # O envelope total funde o decaimento com a onda
            lr_max = lr_max * decay_cosseno * (1.0 + onda)
            
            LP_Fator = 1.0 + lp_kappa * math.cos(lp_omega * math.log(1.0 + progresso * 10.0))
            bakh_thresh_eff = bakh_thresh * LP_Fator
            
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['direcao_buffer'] = torch.empty_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                
                # C++ Fused Kernel Call
                # OMNI logic is pushed to the END inside the C++ Kernel now
                mask_val = int(punning_mask)
                if mask_val > 0x7FFFFFFF:
                    mask_val -= 0x100000000
                    
                if p.is_cpu:
                    self.adamv_cpp.adamv_step_cpu(
                        p, grad, exp_avg, exp_avg_sq, state['direcao_buffer'],
                        lr_max, beta1, beta2, eps, weight_decay,
                        float(progresso), float(bakh_thresh_eff), state['step'], p.numel(),
                        bool(group.get('enable_cooling', False)), bool(group.get('enable_brake', True))
                    )
                    if omni_triggered:
                        if p.dtype == torch.float32:
                            p_int = p.view(torch.int32)
                            sign_exp = p_int & 0xFF800000
                            mant = p_int & 0x007FFFFF
                            scrambled_mant = (((mant + 1) * 31337) & 0x007FFFFF) & mask_val
                            p_new = (sign_exp | scrambled_mant).view(torch.float32)
                            p.copy_(p_new)
                        state['exp_avg'].zero_()
                        state['exp_avg_sq'].mul_(0.1)
                        state['step'] = 0
                elif p.is_cuda and self.adamv_cuda is not None and hasattr(self.adamv_cuda, 'adamv_step_cuda'):
                    self.adamv_cuda.adamv_step_cuda(
                        p, grad, exp_avg, exp_avg_sq, state['direcao_buffer'],
                        lr_max, beta1, beta2, eps, weight_decay,
                        float(progresso), float(bakh_thresh_eff), state['step'], p.numel(),
                        bool(group.get('enable_cooling', False)), bool(group.get('enable_brake', True)),
                        bool(omni_triggered), mask_val
                    )
                    if omni_triggered:
                        state['step'] = 0
                else:
                    # Python fallback para GPU - 100% In-Place BRCM
                    grad_f = grad.float()
                    
                    # Update exp_avg_sq first!
                    exp_avg_sq.mul_(beta2).addcmul_(grad_f, grad_f, value=1.0 - beta2)
                    
                    bias_correction2 = 1.0 - beta2 ** state['step']
                    v_hat = exp_avg_sq / bias_correction2
                    sqrt_v_hat = v_hat.sqrt()
                    
                    # Excess Shock Isolator (delta)
                    # Tighter tuning: 1.0x margin to prevent excess momentum in sharp minima
                    delta = torch.clamp(torch.abs(grad_f) - 1.5 * sqrt_v_hat, min=0.0)
                    
                    denom_brcm = sqrt_v_hat + delta + eps
                    bakh_residual = (delta * delta) / (denom_brcm * 2.0)
                    
                    # Full curvature shift to avoid carrying too much momentum
                    curvature_shift = bakh_residual / (sqrt_v_hat + eps)
                    beta1_eff = torch.exp(-0.5 * curvature_shift) * beta1
                    
                    bias_correction1_dynamic = 1.0 - beta1_eff ** state['step']
                    
                    # Update momentum In-Place
                    exp_avg.mul_(beta1_eff).add_(grad_f * (1.0 - beta1_eff))
                    
                    direcao = (exp_avg / bias_correction1_dynamic) / (sqrt_v_hat + eps)
                    
                    norm_dir = torch.linalg.norm(direcao) / math.sqrt(p.numel())
                    if group.get('enable_cooling', False):
                        envelope = (1.0 + progresso) / (progresso + norm_dir + eps)
                        if progresso < 0.1:
                            cooling = 0.01 + (progresso / 0.1) * 0.99
                        else:
                            cos_progresso = (progresso - 0.1) / 0.9
                            cooling = 0.01 + 0.5 * 0.99 * (1.0 + math.cos(math.pi * cos_progresso))
                        lr_efetivo = lr_max * torch.clamp(envelope * cooling, max=1.5)
                    else:
                        lr_efetivo = lr_max
                    
                    a = direcao.mul_(lr_efetivo)
                    sqrt_v_hat = v_hat.sqrt()
                    
                    explosao_mask = torch.abs(grad) > (bakh_thresh_eff * sqrt_v_hat)
                    
                    # Dimensional Mismatch Fix: parameter absolute magnitude
                    # Tightened Tuning: Stronger brake penalty for late-stage stability
                    denom = p.abs() + (a.abs() * 2.0) + eps
                    correction = (a * a) / (denom * 2.0)
                    bakhshali_brake = a - (torch.sign(a) * correction)
                    
                    if group.get('enable_brake', True):
                        step_size = torch.where(explosao_mask, bakhshali_brake, a)
                    else:
                        step_size = a
                        
                    # Orthogonal Truncated Levy Flight Injection
                    u = torch.rand_like(p) * 0.999 + 0.0005
                    cauchy = torch.tan(math.pi * (u - 0.5))
                    cauchy = torch.clamp(cauchy, -500.0, 500.0)
                    levy_scale = sqrt_v_hat / (sqrt_v_hat + 0.1)
                    levy_mult = 0.01 * max(0.0, 1.0 - progresso)
                    levy_jump = lr_efetivo * levy_mult * cauchy * levy_scale
                    step_size.sub_(levy_jump)
                        
                    if weight_decay != 0:
                        wd_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                        p.mul_(1.0 - lr_max * weight_decay * wd_factor)
                        
                    p.sub_(step_size)
                    
                    if omni_triggered:
                        if p.dtype == torch.float32:
                            p_int = p.view(torch.int32)
                            sign_exp = p_int & 0xFF800000
                            mant = p_int & 0x007FFFFF
                            
                            scrambled_mant = (((mant + 1) * 31337) & 0x007FFFFF) & mask_val
                            p_new = (sign_exp | scrambled_mant).view(torch.float32)
                            p.copy_(p_new)
                        
                        state['exp_avg'].zero_()
                        state['exp_avg_sq'].mul_(0.1)
                        state['step'] = 0

        if omni_triggered:
            g_state['omni_clock_reset_step'] = current_step
                        
        return loss


### 3. The Benchmark Setup (Tartarus Arena)
We will test AdamV against AdamW on a Tartarus MLP (100 layers, no residual connections) designed to suffer from catastrophic vanishing gradients.

In [ ]:
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

class TartarusMLP(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=512, num_layers=100, num_classes=10):
        super().__init__()
        layers = []
        for i in range(num_layers):
            in_f = input_dim if i == 0 else hidden_dim
            layers.append(nn.Linear(in_f, hidden_dim))
            layers.append(nn.GELU())
        layers.append(nn.Linear(hidden_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [ ]:
def run_tartarus_arena():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Running on {device}')
    
    N = 20000
    X = torch.randn(N, 512).to(device)
    # Create a real mathematical signal for the 10 classes
    signal = torch.sin(X[:, 0] * 5.0) + torch.cos(X[:, 1] * 5.0) + X[:, 2:10].sum(dim=1)
    # Use modulo to create perfectly balanced classes (0 to 9) with high non-linearity
    Y = (torch.abs(signal * 10).long()) % 10
    
    # Add 50% destructive noise
    noise_idx = torch.randperm(N)[:int(0.5 * N)]
    Y[noise_idx] = torch.randint(0, 10, (len(noise_idx),)).to(device)
    
    optimizers = ['AdamW', 'AdamV']
    results = {}
    
    import gc
    for opt_name in optimizers:
        torch.cuda.empty_cache()
        gc.collect()
        torch.manual_seed(42)
        model = TartarusMLP().to(device)
        
        if opt_name == 'AdamW':
            opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
        else:
            opt = AdamVCpp(model.parameters(), lr=1e-3, weight_decay=0.01)
            
        criterion = nn.CrossEntropyLoss()
        
        history = []
        start_time = time.time()
        
        for step in range(2000):
            # Mini-batch massivo para saturar a GPU (Reduzindo overhead de CPU)
            idx = torch.randint(0, 20000, (4096,), device=device)
            X_batch = X[idx]
            Y_batch = Y[idx]
            
            opt.zero_grad()
            out = model(X_batch)
            loss = criterion(out, Y_batch)
            loss.backward()
            
            if opt_name == 'AdamW':
                opt.step()
            else:
                opt.step(current_loss=loss.item())
                
            history.append(loss.item())
            
            if step % 200 == 0 or step == 1999:
                print(f'[{opt_name}] Step {step}/2000 | Loss: {loss.item():.4f}')
            
        results[opt_name] = history
        print(f'{opt_name} finished in {time.time()-start_time:.2f}s | Final Loss: {history[-1]:.4f}')
        
    plt.figure(figsize=(10,6))
    for name, hist in results.items():
        plt.plot(hist, label=name)
    plt.title('Tartarus Arena (100-Layer Deep MLP w/ 50% Noise)')
    plt.xlabel('Steps')
    plt.ylabel('CrossEntropy Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

run_tartarus_arena()
